In [19]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Set Plotly theme
pio.templates.default = "plotly_white"

print("Plotly version:", __import__("plotly").__version__)

# Load our Placement Skill Gap Analytics dataset (using the correct available filename)
df = pd.read_csv("student_academic_placement_performance_dataset(1).csv")

# Display dataset information
print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())

print("\nFirst 5 Records:")
print(df.head())

Plotly version: 5.24.1
Dataset Shape: (5120, 18)

Column Names:
['student_id', 'gender', 'ssc_percentage', 'hsc_percentage', 'degree_percentage', 'cgpa', 'entrance_exam_score', 'technical_skill_score', 'soft_skill_score', 'internship_count', 'live_projects', 'work_experience_months', 'certifications', 'attendance_percentage', 'backlogs', 'extracurricular_activities', 'placement_status', 'salary_package_lpa']

First 5 Records:
   student_id  gender  ssc_percentage  hsc_percentage  degree_percentage  \
0           1    Male            53.0            79.0               56.0   
1           2  Female            56.0            54.0               59.0   
2           3    Male            94.0            83.0               83.0   
3           4    Male            84.0            71.0               87.0   
4           5    Male            58.0            88.0               74.0   

   cgpa  entrance_exam_score  technical_skill_score  soft_skill_score  \
0  8.87                 50.0            

### Check for Duplicate Records

In [20]:
# 1. Check for completely identical rows across all columns
total_duplicates = df.duplicated().sum()
print(f"Total completely duplicate rows: {total_duplicates}")

# 2. Check for duplicate Student IDs (which should be unique)
unique_id_duplicates = df.duplicated(subset=['student_id']).sum()
print(f"Duplicate Student IDs: {unique_id_duplicates}")

# 3. If duplicates exist, display them
if total_duplicates > 0:
    print("\nFirst few duplicate rows:")
    display(df[df.duplicated(keep=False)].head())
else:
    print("\nNo duplicate rows detected in the dataset.")

Total completely duplicate rows: 111
Duplicate Student IDs: 120

First few duplicate rows:


,student_id,gender,ssc_percentage,hsc_percentage,degree_percentage,cgpa,entrance_exam_score,technical_skill_score,soft_skill_score,internship_count,live_projects,work_experience_months,certifications,attendance_percentage,backlogs,extracurricular_activities,placement_status,salary_package_lpa
34,35,Female,84.0,71.0,75.0,8.45,89.0,40.0,44.0,0.0,0.0,13.0,4.0,72.0,0.0,Yes,0.0,0.00
64,65,Male,58.0,68.0,62.0,6.23,64.0,74.0,95.0,4.0,4.0,18.0,5.0,91.0,4.0,No,0.0,0.00
130,131,Female,73.0,75.0,76.0,9.49,66.0,63.0,69.0,2.0,4.0,23.0,4.0,97.0,1.0,No,1.0,9.97
161,162,Male,55.0,84.0,80.0,6.40,62.0,89.0,41.0,4.0,3.0,6.0,5.0,99.0,0.0,No,0.0,0.00
189,190,Female,74.0,60.0,75.0,5.78,66.0,56.0,50.0,2.0,2.0,4.0,5.0,66.0,0.0,Yes,0.0,0.00


### Check for Missing Values

In [21]:
# 1. Compute missing values per column
missing_count = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

# 2. Combine into a summary DataFrame
missing_summary = pd.DataFrame({
    'Missing Values': missing_count,
    'Percentage (%)': missing_percentage
})

# 3. Filter to display only columns with missing values (if any), or display the whole summary
print(f"Total missing values in the dataset: {df.isnull().sum().sum()}")
display(missing_summary)

Total missing values in the dataset: 150


,Missing Values,Percentage (%)
student_id,0,0.000000
gender,8,0.156250
ssc_percentage,8,0.156250
hsc_percentage,9,0.175781
degree_percentage,14,0.273438
cgpa,11,0.214844
entrance_exam_score,10,0.195312
technical_skill_score,2,0.039062
soft_skill_score,9,0.175781
internship_count,7,0.136719


### Replace Missing Values (Imputation)

In [22]:
# 1. Identify numeric and categorical columns
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns

# 2. Impute missing values
# Fill numeric columns with their respective median
for col in numeric_cols:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"Imputed numeric column '{col}' with median: {median_val}")

# Fill categorical columns with their respective mode
for col in categorical_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f"Imputed categorical column '{col}' with mode: {mode_val}")

# 3. Verify that no missing values remain
print(f"\nRemaining missing values after imputation: {df.isnull().sum().sum()}")

Imputed numeric column 'ssc_percentage' with median: 72.0
Imputed numeric column 'hsc_percentage' with median: 72.0
Imputed numeric column 'degree_percentage' with median: 72.0
Imputed numeric column 'cgpa' with median: 7.69
Imputed numeric column 'entrance_exam_score' with median: 69.0
Imputed numeric column 'technical_skill_score' with median: 70.0
Imputed numeric column 'soft_skill_score' with median: 69.0
Imputed numeric column 'internship_count' with median: 2.0
Imputed numeric column 'live_projects' with median: 3.0
Imputed numeric column 'work_experience_months' with median: 12.0
Imputed numeric column 'certifications' with median: 2.0
Imputed numeric column 'attendance_percentage' with median: 80.0
Imputed numeric column 'backlogs' with median: 3.0
Imputed numeric column 'placement_status' with median: 0.0
Imputed numeric column 'salary_package_lpa' with median: 0.0
Imputed categorical column 'gender' with mode: Female
Imputed categorical column 'extracurricular_activities' wit

**HISTOGRAM: Distribution of Student CGPA**

In [23]:
# Choose the column you want to plot
target_column = 'entrance_exam_score'  # Try changing this to 'salary_package_lpa' or 'attendance_percentage'

fig_hist = px.histogram(
    df,
    x=target_column,
    color="placement_status",
    marginal="box",  # Adds a box plot on top to see outliers/medians
    title=f"Distribution of {target_column.replace('_', ' ').title()} by Placement Status",
    labels={
        target_column: target_column.replace('_', ' ').title(),
        "count": "Student Count"
    },
    barmode="overlay",
    opacity=0.75,
    color_discrete_map={0: "#EF553B", 1: "#636EFA"}
)

fig_hist.update_layout(
    template="plotly_white",
    bargap=0.1,
    title_x=0.5,
    legend_title_text="Placement Status (0=Unplaced, 1=Placed)"
)

fig_hist.show()

**2. SCATTER PLOT: CGPA vs Placement Status**

In [24]:
scatter_df = df.copy()
scatter_df["Placement Status"] = scatter_df["placement_status"].map({0: "Not Placed", 1: "Placed"})

# Generate a multi-dimensional scatter plot
fig2 = px.scatter(
    scatter_df,
    x="cgpa",
    y="entrance_exam_score",
    color="Placement Status",
    size="salary_package_lpa",
    hover_name="gender",
    hover_data={
        "cgpa": ":.2f",
        "entrance_exam_score": ":.1f",
        "salary_package_lpa": ":.2f lpa",
        "Placement Status": True
    },
    color_discrete_map={
        "Not Placed": "#EF553B",
        "Placed": "#636EFA"
    },
    opacity=0.7,
    title="Academic Performance vs Entrance Exam Performance",
    labels={
        "cgpa": "CGPA",
        "entrance_exam_score": "Entrance Exam Score",
        "salary_package_lpa": "Salary Package (LPA)"
    }
)

fig2.update_layout(
    template="plotly_white",
    width=900,
    height=600,
    title_x=0.5,
    legend_title_text="Placement Outcome",
    margin=dict(l=60, r=40, t=80, b=60)
)

fig2.show()

**box plot**

In [25]:
# Map numeric placement_status to descriptive text for plotting
plot_df = df.copy()
plot_df["placement_status_text"] = plot_df["placement_status"].map({0: "Not Placed", 1: "Placed"})

fig = px.box(
    plot_df,
    x="placement_status_text",
    y="technical_skill_score",
    color="placement_status_text",
    category_orders={"placement_status_text": ["Not Placed", "Placed"]},
    color_discrete_map={
        "Not Placed": "#8CB9E8",
        "Placed": "#238B68"
    },
    title="Technical Skill Scores by Placement Status",
    labels={
        "placement_status_text": "Placement Status",
        "technical_skill_score": "Technical Skill Score"
    },
    points=False
)

fig.update_layout(
    template="plotly_white",
    width=800,
    height=500,
    showlegend=False,
    title_x=0.5,
    margin=dict(l=60, r=30, t=80, b=60)
)

fig.update_yaxes(range=[0, 100], dtick=20)
fig.show()

### Dynamic Histogram of Numeric Features

### 4. Interactive Scatter Plot: CGPA vs Entrance Exam Score by Placement & Salary